# AgriVoice: N-ATLaS on Kaggle

Runs the official [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) on Kaggle's free GPUs and connects it to the AgriVoice backend.

## One-time setup
1. **Settings → Accelerator → GPU T4 x2** (N-ATLaS needs both GPUs).
2. **Settings → Internet → On** (requires a phone-verified Kaggle account).
3. **Add-ons → Secrets**, add and attach to this notebook:
   - `HF_TOKEN`: Hugging Face token of an account that accepted the N-ATLaS licence
   - `AGRIVOICE_URL`: your backend, e.g. `https://your-username-agrivoice.hf.space` *(optional)*
   - `AGRIVOICE_ADMIN_TOKEN`: the backend's `ADMIN_TOKEN` *(optional)*

   Secrets only work for runs started from the Kaggle website. For runs started
   through the Kaggle API, put the same values in a **private** dataset named
   `agrivoice-secrets`, one file per secret (`hf_token`, `agrivoice_url`,
   `agrivoice_admin_token`), and attach it to this notebook.

## Each session
- **Easiest:** press **Start** in the AgriVoice admin dashboard (N-ATLaS tab). It runs this notebook for you and **Stop** shuts it down. Needs the private `agrivoice-secrets` dataset.
- **By hand:** *Save Version → Save & Run All (Commit)*. Set `RUN_HOURS` in the last cell first (default 2; ~9 on review days).

The last cell prints the public N-ATLaS link and registers it with the backend. Without backend settings, paste the link into the backend's `/admin` page.

In [ ]:
# Filled in by the AgriVoice backend when it starts this notebook from /admin.
# Leave it empty for runs started by hand (Kaggle Secrets are used instead).
RUN_CONFIG = {}

In [ ]:
!pip install -q "gradio==6.29.0" spaces "transformers==5.17.0" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

import glob

_secrets = UserSecretsClient()

def _secret(name):
    # 1) Kaggle Secrets: only reachable when the run is started from the website
    try:
        value = _secrets.get_secret(name)
        if value:
            return value
    except Exception:
        pass
    # 2) Private dataset "agrivoice-secrets" holding one file per secret,
    #    used for runs started through the Kaggle API
    for path in glob.glob(f"/kaggle/input/**/agrivoice-secrets/{name.lower()}", recursive=True):
        value = open(path).read().strip()
        if value:
            return value
    return None

os.environ["HF_TOKEN"] = _secret("HF_TOKEN") or ""
os.environ["DEVICE"] = "auto"  # spread N-ATLaS across both T4 GPUs
AGRIVOICE_URL = (RUN_CONFIG.get("backend_url") or _secret("AGRIVOICE_URL") or "").rstrip("/")
AGRIVOICE_ADMIN_TOKEN = None if RUN_CONFIG else _secret("AGRIVOICE_ADMIN_TOKEN")
assert os.environ["HF_TOKEN"], "Add the HF_TOKEN secret, or the private agrivoice-secrets dataset."
print("Started from:", "AgriVoice dashboard" if RUN_CONFIG else "Kaggle website",
      "| Backend:", AGRIVOICE_URL or "none")

In [ ]:
%%writefile app.py
"""
N-ATLaS on Hugging Face ZeroGPU.

Follows the official usage on https://huggingface.co/NCAIR1/N-ATLaS:
  - AutoTokenizer + AutoModelForCausalLM, float16
  - apply_chat_template(..., tokenize=False, date_string=<today>)
  - tokenizer(text, add_special_tokens=False)
  - generate(max_new_tokens, use_cache=True, repetition_penalty=1.12, greedy)

Two differences from the model card's snippet, both needed to run it:
  - `from datetime import datetime` (missing in the card)
  - decode only the newly generated tokens, so the prompt isn't echoed back

The AgriVoice backend calls the `/generate` endpoint with a list of chat
messages. The "Chat" tab is for trying the model by hand.
"""
from __future__ import annotations

import os
from datetime import datetime

import gradio as gr
import spaces
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "NCAIR1/N-ATLaS")
# "cuda": ZeroGPU (model placed on CUDA at import time, as ZeroGPU requires)
# "auto": spread across several GPUs, e.g. Kaggle's 2 × T4 (16 GB doesn't fit one)
# "cpu":  local testing with a small stand-in model only
DEVICE = os.getenv("DEVICE", "cuda")
HF_TOKEN = os.getenv("HF_TOKEN")  # Space secret; N-ATLaS is a gated model

DEFAULT_MAX_NEW_TOKENS = 512
MAX_NEW_TOKENS_LIMIT = 1000  # The model card's value
REPETITION_PENALTY = 1.12

DEFAULT_SYSTEM_PROMPT = (
    "you are a large language model trained by Awarri AI technologies. "
    "You are a friendly assistant and you are here to help."
)

_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if _tokenizer is None:
    raise RuntimeError(f"Could not load the tokenizer for {MODEL_ID}")
tokenizer = _tokenizer

if DEVICE == "auto":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, token=HF_TOKEN, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32 if DEVICE == "cpu" else torch.float16,
        token=HF_TOKEN,
    ).to(DEVICE)
model.eval()


def format_text_for_inference(messages: list[dict[str, str]]) -> str:
    current_date = datetime.now().strftime("%d %b %Y")
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        date_string=current_date,
    )
    if not isinstance(text, str):  # tokenize=False always returns a string
        raise TypeError(f"Unexpected chat template output: {type(text).__name__}")
    return text


@spaces.GPU(duration=90)
def generate(messages: list[dict[str, str]], max_new_tokens: float = DEFAULT_MAX_NEW_TOKENS) -> str:
    """
    Generate the assistant's reply.

    Args:
        messages: [{"role": "system" | "user" | "assistant", "content": "..."}, ...]
        max_new_tokens: Upper bound on reply length (capped at 1000). A float because
            Gradio sliders and API clients may send e.g. 512.0.
    """
    if not messages or not all(
        isinstance(m, dict) and m.get("role") and isinstance(m.get("content"), str)
        for m in messages
    ):
        raise gr.Error("`messages` must be a list of {role, content} objects.")

    max_new_tokens = max(1, min(int(max_new_tokens), MAX_NEW_TOKENS_LIMIT))
    text = format_text_for_inference(messages)
    input_tokens = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **input_tokens,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            repetition_penalty=REPETITION_PENALTY,
            # The card passes temperature=0.1 without do_sample, which decodes
            # greedily; this is the same behaviour stated explicitly.
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_tokens["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    if not isinstance(reply, str):  # one sequence decodes to one string
        raise TypeError(f"Unexpected decode output: {type(reply).__name__}")
    return reply.strip()


def _text(content: object) -> str:
    """Gradio 6 chat history may hold content as a list of parts."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            part.get("text", "") if isinstance(part, dict) else str(part) for part in content
        )
    return str(content)


def chat(message: str, history: list[dict]) -> str:
    messages = [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT}]
    messages += [{"role": m["role"], "content": _text(m["content"])} for m in history]
    messages.append({"role": "user", "content": message})
    return generate(messages)


with gr.Blocks(title="N-ATLaS") as demo:
    gr.Markdown("# N-ATLaS\nNCAIR's multilingual model (Yoruba, Hausa, Igbo, English).")
    with gr.Tab("Chat"):
        gr.ChatInterface(chat, api_visibility="private")
    with gr.Tab("API"):
        messages_in = gr.JSON(
            label="messages",
            value=[
                {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                {"role": "user", "content": "menene ake nufi da gwagwarmaya"},
            ],
        )
        max_tokens_in = gr.Slider(1, MAX_NEW_TOKENS_LIMIT, value=DEFAULT_MAX_NEW_TOKENS, step=1,
                                  label="max_new_tokens")
        reply_out = gr.Textbox(label="reply")
        gr.Button("Generate").click(
            generate, inputs=[messages_in, max_tokens_in], outputs=reply_out, api_name="generate"
        )

if __name__ == "__main__":
    demo.launch()

In [ ]:
import time
import requests

# How long to keep N-ATLaS online (runs started by hand). Kaggle gives 30 GPU
# hours a week and stops a session after ~9-12 hours anyway.
RUN_HOURS = float(RUN_CONFIG.get("run_hours", 2))
RUN_KEY = RUN_CONFIG.get("run_key")

import app  # downloads N-ATLaS (~16 GB) on first run and loads it onto the GPUs

_, local_url, share_url = app.demo.launch(share=True, prevent_thread_lock=True)
print("\nN-ATLaS public link:", share_url)


def now():
    return time.strftime("%H:%M:%S")


def notify(event):
    """Dashboard-started runs: tell the backend; returns its JSON reply or None."""
    try:
        r = requests.post(
            f"{AGRIVOICE_URL}/api/v1/natlas/{event}",
            json={"run_key": RUN_KEY, "endpoint": share_url},
            timeout=30,
        )
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        print(now(), f"Could not reach AgriVoice ({event}):", exc)
        return None


def register_by_hand():
    if not (AGRIVOICE_URL and AGRIVOICE_ADMIN_TOKEN):
        print("Paste this link into your backend's /admin page.")
        return
    try:
        r = requests.put(
            f"{AGRIVOICE_URL}/api/v1/admin/llm-endpoint",
            json={"endpoint": share_url},
            headers={"Authorization": f"Bearer {AGRIVOICE_ADMIN_TOKEN}"},
            timeout=30,
        )
        r.raise_for_status()
        print(now(), "Registered with AgriVoice:", r.json()["endpoint"])
    except Exception as exc:
        print(now(), "Could not register with AgriVoice:", exc)


stop_at = time.time() + RUN_HOURS * 3600
reason = f"reached {RUN_HOURS:g} h"
try:
    if RUN_KEY:
        notify("register") and print(now(), "Registered with AgriVoice")
        # Check in every minute; the dashboard's Stop button answers "stop"
        while time.time() < stop_at:
            time.sleep(60)
            reply = notify("heartbeat")
            if reply and reply.get("action") == "stop":
                reason = "stopped from the dashboard"
                break
    else:
        register_by_hand()
        # Keep the session alive (and re-register in case the backend restarted)
        while time.time() < stop_at:
            time.sleep(min(600, max(1, stop_at - time.time())))
            register_by_hand()
finally:
    if RUN_KEY:
        notify("stopped")
    app.demo.close()
    print(now(), f"N-ATLaS is now offline ({reason}).")